# Data Preprocessing

## 1. Introduction

Data preprocessing is a critical step in the data science workflow that prepares raw data for analysis and predictive modeling. Real-world datasets often contain differences in data formats, temporal resolution, and variable representations, making preprocessing essential before integrating multiple data sources.

In this study, the Air Pollution, Fire, Wind, and Weather Datasets are transformed into a consistent analytical format through a series of preprocessing steps. These include data type conversion, extraction of the Delhi study region where applicable, temporal aggregation, feature preparation, and dataset integration. The resulting processed dataset provides a reliable foundation for exploratory data analysis, feature engineering, and machine learning in the subsequent stages of the project.

## 2. Objectives

The objectives of this notebook are to:

- Prepare each dataset for analysis by converting variables into appropriate formats.
- Clean and transform the Air Pollution, Fire, Wind, and Weather Datasets.
- Extract the Delhi study region from the Wind and Weather Datasets.
- Aggregate the Fire, Wind, and Weather Datasets to daily observations.
- Integrate all datasets into a unified analytical dataset.
- Generate the final processed dataset for exploratory data analysis and predictive modeling.

## 3. Required Libraries

In [71]:
import pandas as pd
import numpy as np

## 4. Loading the Raw Datasets

### 4.1 Loading the Datasets

The four raw datasets are loaded into pandas DataFrames to begin the preprocessing stage. At this point, no transformations or cleaning operations are performed. Each dataset is imported in its original form to preserve the raw data before applying preprocessing techniques in the subsequent sections.

In [72]:
aqi_df = pd.read_csv(r"city_day/city_day.csv")
fire_df = pd.read_csv("DL_FIRE_SV-C2_774473/fire_archive_SV-C2_774473.csv")
wind_df = pd.read_csv("wind_RAW.csv")
weather_df = pd.read_csv("weather_RAW.csv")

### 4.2 Verifying Dataset Dimensions

The dimensions of each dataset are examined to confirm that the datasets have been loaded successfully. This verification ensures that no records were lost or altered during the loading process and establishes the baseline sizes of the raw datasets before preprocessing.

In [73]:
datasets = {
    "Air Pollution": aqi_df,
    "Fire": fire_df,
    "Wind": wind_df,
    "Weather": weather_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Air Pollution: (29531, 16)
Fire: (562486, 15)
Wind: (2533952, 9)
Weather: (2955892, 9)


### Findings

All four datasets were successfully loaded into pandas DataFrames without any errors. The datasets retain their original dimensions, comprising **29,531** air pollution records, **562,486** fire observations, **2,533,952** wind observations, and **2,955,892** weather observations. These raw datasets serve as the starting point for the preprocessing steps performed in this notebook.

## 5. Air Pollution Dataset Preprocessing

### 5.1 Converting the Date Variable

The **Date** variable is converted from a character data type to the datetime format to enable efficient time-based operations such as sorting, filtering, merging, and feature engineering. Using the datetime format also ensures consistency with the other datasets before integration.

In [74]:
aqi_df["Date"] = pd.to_datetime(aqi_df["Date"])

print(aqi_df["Date"].dtype)

datetime64[ns]


### Findings

The **Date** variable was successfully converted to the **datetime64[ns]** data type. This enables efficient time-based operations, including chronological sorting, temporal filtering, dataset integration, and the creation of time-related features during the subsequent preprocessing stages.

### 5.2 Sorting the Dataset by Date

The Air Pollution Dataset is sorted in chronological order based on the **Date** variable. Maintaining chronological order is important for time-series analysis, feature engineering, and the creation of lag variables in later stages of the project.

In [75]:
aqi_df = aqi_df.sort_values("Date").reset_index(drop=True)

aqi_df[["Date"]].head()

,Date
0,2015-01-01
1,2015-01-01
2,2015-01-01
3,2015-01-01
4,2015-01-01


### Findings

The Air Pollution Dataset was successfully sorted in chronological order based on the **Date** variable. The earliest records now appear first, ensuring that the dataset is correctly ordered for time-series analysis, temporal integration with the other datasets, and the generation of lag-based features in the later stages of the project.

In [76]:
aqi_df.columns.tolist()

['City',
 'Date',
 'PM2.5',
 'PM10',
 'NO',
 'NO2',
 'NOx',
 'NH3',
 'CO',
 'SO2',
 'O3',
 'Benzene',
 'Toluene',
 'Xylene',
 'AQI',
 'AQI_Bucket']

### 5.3 Verifying the AQI Dataset

After converting the **Date** variable and arranging the records chronologically, the structure of the Air Pollution Dataset is verified to ensure that the preprocessing steps have been applied successfully. This verification confirms that the dataset is ready for integration with the remaining datasets.

In [77]:
print("Shape:", aqi_df.shape)

print("\nData Types:")
print(aqi_df.dtypes)

print("\nFirst Five Records:")
display(aqi_df.head())

Shape: (29531, 16)

Data Types:
City                  object
Date          datetime64[ns]
PM2.5                float64
PM10                 float64
NO                   float64
NO2                  float64
NOx                  float64
NH3                  float64
CO                   float64
SO2                  float64
O3                   float64
Benzene              float64
Toluene              float64
Xylene               float64
AQI                  float64
AQI_Bucket            object
dtype: object

First Five Records:


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Chennai,2015-01-01,NaN,NaN,16.30,15.39,22.68,4.59,1.17,9.20,11.35,0.17,NaN,NaN,NaN,NaN
2,Delhi,2015-01-01,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,41.68,14.36,24.86,9.84,472.0,Severe
3,Lucknow,2015-01-01,NaN,NaN,2.11,13.46,4.57,NaN,12.15,169.57,25.92,1.35,3.93,NaN,NaN,NaN
4,Mumbai,2015-01-01,NaN,NaN,NaN,NaN,27.38,NaN,0.00,NaN,NaN,0.00,0.00,0.00,NaN,NaN


### Findings

The Air Pollution Dataset has been successfully prepared for the subsequent preprocessing stages. The **Date** variable is stored in the appropriate datetime format, the records are arranged chronologically, and all variables have been retained in their original form. The dataset is now ready to be integrated with the Fire, Wind, and Weather Datasets after their respective preprocessing steps.

### 5.4 Extracting Delhi Air Pollution Data

The Air Pollution Dataset contains observations for multiple Indian cities. Since the objective of this study is to investigate the factors influencing **Delhi's air quality**, only records corresponding to **Delhi** are retained. Restricting the dataset to the study area ensures consistency with the Fire, Wind, and Weather Datasets that will also be processed for the Delhi region.

In [78]:
aqi_df = aqi_df[aqi_df["City"] == "Delhi"].copy()

print("Shape:", aqi_df.shape)

display(aqi_df.head())

Shape: (2009, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
2,Delhi,2015-01-01,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,41.68,14.36,24.86,9.84,472.0,Severe
6,Delhi,2015-01-02,186.18,269.55,62.09,32.87,88.14,31.83,9.54,6.65,29.97,10.55,20.09,4.29,454.0,Severe
12,Delhi,2015-01-03,87.18,131.90,25.73,30.31,47.95,69.55,10.61,2.65,19.71,3.91,10.23,1.99,143.0,Moderate
23,Delhi,2015-01-04,151.84,241.84,25.01,36.91,48.62,130.36,11.54,4.63,25.36,4.26,9.71,3.34,319.0,Very Poor
27,Delhi,2015-01-05,146.60,219.13,14.01,34.92,38.25,122.88,9.20,3.33,23.20,2.80,6.21,2.96,325.0,Very Poor


### Findings

The Air Pollution Dataset was successfully filtered to retain only observations corresponding to **Delhi**, reducing the dataset from **29,531** records to **2,009** records. The filtered dataset preserves all air quality variables while ensuring that only observations from the study area are included. This preprocessing step aligns the Air Pollution Dataset with the study objective and prepares it for integration with the Fire, Wind, and Weather Datasets.

### 5.5 Selecting the Required AQI Variables

The original air quality dataset contains several pollutant measurements. However, this study focuses on predicting the Air Quality Index (AQI) using external fire activity and meteorological conditions. Therefore, only the variables required for the research objectives are retained. This step reduces unnecessary attributes while preserving the variables needed for exploratory analysis and predictive modelling.

In [79]:
aqi_df = aqi_df[
    [
        "Date",
        "AQI",
        "AQI_Bucket",
        "PM2.5"
    ]
].copy()

print("Shape:", aqi_df.shape)

display(aqi_df.head())

Shape: (2009, 4)


,Date,AQI,AQI_Bucket,PM2.5
2,2015-01-01,472.0,Severe,313.22
6,2015-01-02,454.0,Severe,186.18
12,2015-01-03,143.0,Moderate,87.18
23,2015-01-04,319.0,Very Poor,151.84
27,2015-01-05,325.0,Very Poor,146.60


### 5.6 Verifying the Study Period

The temporal coverage of the filtered Delhi Air Pollution Dataset is verified to ensure that it spans the intended study period. Confirming the earliest and latest observations helps determine whether the dataset is suitable for integration with the remaining datasets.

In [80]:
print("Earliest Date :", aqi_df["Date"].min())
print("Latest Date   :", aqi_df["Date"].max())

Earliest Date : 2015-01-01 00:00:00
Latest Date   : 2020-07-01 00:00:00


### Findings

The filtered Delhi Air Pollution Dataset spans from **1 January 2015** to **1 July 2020**, providing more than five years of continuous air quality observations. This period is consistent with the study timeframe and is suitable for integration with the Fire, Wind, and Weather Datasets during the subsequent preprocessing stages.

## 6. Fire Dataset Preprocessing

### 6.1 Converting the Acquisition Date

The Fire Dataset records the acquisition date of each detected fire event in the **acq_date** variable. This variable is converted to the datetime format to facilitate temporal aggregation, dataset integration, and feature engineering. Consistent datetime formatting is essential for merging the Fire Dataset with the Air Pollution, Wind, and Weather Datasets.

In [81]:
fire_df["acq_date"] = pd.to_datetime(fire_df["acq_date"])

print(fire_df["acq_date"].dtype)

datetime64[ns]


### 6.2 Sorting the Fire Dataset by Date

The Fire Dataset is sorted in chronological order based on the **acq_date** variable. Maintaining chronological order ensures consistency during daily aggregation and facilitates accurate temporal integration with the remaining datasets.

In [82]:
fire_df = fire_df.sort_values("acq_date").reset_index(drop=True)

fire_df[["acq_date"]].head()

,acq_date
0,2015-01-01
1,2015-01-01
2,2015-01-01
3,2015-01-01
4,2015-01-01


### 6.3 Examining the Fire Dataset Structure

The variables available in the Fire Dataset are examined before feature engineering. Understanding the available attributes helps identify the variables required to construct daily fire-related indicators that will later be integrated with the Air Pollution Dataset.

In [83]:
print("Shape:", fire_df.shape)

print("\nColumns:")
print(fire_df.columns.tolist())

print("\nFirst Five Records:")
display(fire_df.head())

Shape: (562486, 15)

Columns:
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']

First Five Records:


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,28.98106,77.89695,330.35,0.50,0.41,2015-01-01,744,SNPP,SNPP,n,2,280.03,3.30,D,0
1,29.90940,74.95330,295.53,0.60,0.70,2015-01-01,2149,SNPP,SNPP,n,2,281.40,0.95,N,2
2,31.26913,77.39223,305.97,0.76,0.77,2015-01-01,2148,SNPP,SNPP,n,2,274.47,3.47,N,0
3,29.91004,74.94714,302.40,0.40,0.60,2015-01-01,2008,SNPP,SNPP,n,2,282.21,0.86,N,2
4,31.66629,75.61075,339.54,0.39,0.44,2015-01-01,745,SNPP,SNPP,n,2,296.41,4.84,D,0


### Findings

The Fire Dataset contains **562,486** fire observations and **15** variables describing the location, intensity, detection time, and characteristics of each fire event. Important variables for this study include **acq_date**, **latitude**, **longitude**, and **frp**, which will be used to generate daily fire-related indicators for integration with the Air Pollution Dataset.

### 6.4 Aggregating Daily Fire Observations

The Fire Dataset records individual fire detections. Since the Air Pollution Dataset contains daily observations, the fire records are aggregated on a daily basis to generate summary indicators representing the daily fire activity. These aggregated features provide a consistent temporal resolution for dataset integration.

In [84]:
fire_daily = (
    fire_df.groupby("acq_date")
    .agg(
        Fire_Count=("frp", "count"),
        Total_FRP=("frp", "sum"),
        Mean_FRP=("frp", "mean"),
        Max_FRP=("frp", "max")
    )
    .reset_index()
    .rename(columns={"acq_date": "Date"})
)

print("Shape:", fire_daily.shape)
display(fire_daily.head())

Shape: (1861, 5)


,Date,Fire_Count,Total_FRP,Mean_FRP,Max_FRP
0,2015-01-01,30,129.63,4.321000,11.74
1,2015-01-02,15,84.92,5.661333,16.66
2,2015-01-03,4,5.93,1.482500,2.07
3,2015-01-04,15,81.51,5.434000,15.46
4,2015-01-05,34,110.50,3.250000,8.84


### Findings

The individual fire observations were successfully aggregated into **1,861 daily records**. Four daily fire-related indicators were generated: **Fire_Count**, **Total_FRP**, **Mean_FRP**, and **Max_FRP**. This aggregation converts the raw fire detections into a daily representation, making the Fire Dataset compatible with the daily Air Pollution Dataset for subsequent integration.

### 6.5 Verifying the Study Period

The temporal coverage of the aggregated Fire Dataset is verified to ensure that it aligns with the study period and is suitable for integration with the Air Pollution Dataset.

In [85]:
print("Earliest Date :", fire_daily["Date"].min())
print("Latest Date   :", fire_daily["Date"].max())

Earliest Date : 2015-01-01 00:00:00
Latest Date   : 2020-07-01 00:00:00


### Findings

The aggregated Fire Dataset spans from **1 January 2015** to **1 July 2020**, matching the temporal coverage of the Delhi Air Pollution Dataset. This consistency ensures that the fire-related indicators can be accurately integrated with the air quality observations during the dataset merging process.

## 7. Wind Dataset Preprocessing

### 7.1 Converting Date and Time Variables

The Wind Dataset contains temporal information in the **valid_time** variable. This variable is converted to the datetime format to enable chronological ordering, temporal filtering, daily aggregation, and integration with the remaining datasets. Proper datetime formatting is essential for time-series preprocessing.

In [86]:
wind_df["valid_time"] = pd.to_datetime(wind_df["valid_time"])

print(wind_df["valid_time"].dtype)

datetime64[ns]


### 7.2 Examining the Wind Dataset Structure

The structure of the Wind Dataset is examined to verify the available variables and confirm that the latitude, longitude, and temporal information required for spatial filtering and daily aggregation are present.

In [87]:
print("Shape:", wind_df.shape)

print("\nColumns:")
print(wind_df.columns.tolist())

print("\nFirst Five Records:")
display(wind_df.head())

Shape: (2533952, 9)

Columns:
['time', 'latitude', 'longitude', 'number', 'step', 'surface', 'valid_time', 'u10', 'v10']

First Five Records:


,time,latitude,longitude,number,step,surface,valid_time,u10,v10
0,2015-01-01 00:00:00,32.0,74.00,0,0 days,0.0,2015-01-01,-0.831787,-1.742523
1,2015-01-01 00:00:00,32.0,74.25,0,0 days,0.0,2015-01-01,-0.656982,-1.855804
2,2015-01-01 00:00:00,32.0,74.50,0,0 days,0.0,2015-01-01,-0.967529,-2.244476
3,2015-01-01 00:00:00,32.0,74.75,0,0 days,0.0,2015-01-01,-1.294678,-1.374359
4,2015-01-01 00:00:00,32.0,75.00,0,0 days,0.0,2015-01-01,-1.279053,-0.898773


### Findings

The Wind Dataset contains **2,533,952** observations and **9** variables describing the spatial location, temporal information, and horizontal wind components. The **latitude**, **longitude**, and **valid_time** variables will be used for spatial and temporal preprocessing, while the **u10** and **v10** variables will be used to derive daily wind speed for the study area.`

### 7.3 Extracting the Delhi Study Region

The Wind Dataset covers a large geographical area. To ensure consistency with the study objective, only observations corresponding to the Delhi study region are retained. Spatial filtering reduces the dataset size and preserves only the wind information relevant to Delhi before daily aggregation.

In [88]:
wind_delhi = wind_df[
    (wind_df["latitude"] >= 28.0) &
    (wind_df["latitude"] <= 29.0) &
    (wind_df["longitude"] >= 76.5) &
    (wind_df["longitude"] <= 77.5)
].copy()

print("Shape:", wind_delhi.shape)

display(wind_delhi.head())

Shape: (219200, 9)


,time,latitude,longitude,number,step,surface,valid_time,u10,v10
214,2015-01-01 00:00:00,29.0,76.50,0,0 days,0.0,2015-01-01,-0.459717,-2.395843
215,2015-01-01 00:00:00,29.0,76.75,0,0 days,0.0,2015-01-01,-0.262451,-2.479828
216,2015-01-01 00:00:00,29.0,77.00,0,0 days,0.0,2015-01-01,-0.089600,-2.429047
217,2015-01-01 00:00:00,29.0,77.25,0,0 days,0.0,2015-01-01,0.105713,-2.237640
218,2015-01-01 00:00:00,29.0,77.50,0,0 days,0.0,2015-01-01,-0.001709,-1.968109


### Findings

The Wind Dataset was successfully filtered to retain only observations corresponding to the Delhi study region. The dataset size was reduced from **2,533,952** observations to **219,200** observations, substantially decreasing the computational complexity while preserving the wind information relevant to the study area. The filtered dataset is now ready for wind speed calculation and daily aggregation.

### 7.4 Calculating Wind Speed

The Wind Dataset provides the horizontal wind components (`u10` and `v10`) measured at a height of 10 metres. These components are used to calculate the resultant wind speed, which represents the overall wind intensity. The original wind components are retained to preserve the observed wind information for subsequent analysis.

In [89]:
wind_delhi["Wind_Speed"] = np.sqrt(
    wind_delhi["u10"]**2 + wind_delhi["v10"]**2
)

display(
    wind_delhi[
        ["valid_time", "u10", "v10", "Wind_Speed"]
    ].head()
)

,valid_time,u10,v10,Wind_Speed
214,2015-01-01,-0.459717,-2.395843,2.439550
215,2015-01-01,-0.262451,-2.479828,2.493678
216,2015-01-01,-0.089600,-2.429047,2.430699
217,2015-01-01,0.105713,-2.237640,2.240136
218,2015-01-01,-0.001709,-1.968109,1.968110


### Findings

The resultant wind speed was successfully calculated from the horizontal wind components (`u10` and `v10`). The original wind components were retained while adding the derived `Wind_Speed` variable, preserving both the observed wind vectors and the overall wind intensity for subsequent analysis.

### 7.5 Aggregating Daily Wind Data

The Wind Dataset contains multiple observations recorded each day. To match the daily temporal resolution of the other datasets, the wind observations are aggregated by date. Daily mean values of `u10`, `v10`, and wind speed, together with the daily maximum wind speed, are retained to preserve the observed wind characteristics.

In [90]:
wind_daily = (
    wind_delhi
    .assign(Date=wind_delhi["valid_time"].dt.date)
    .groupby("Date")
    .agg(
        Mean_U10=("u10", "mean"),
        Mean_V10=("v10", "mean"),
        Mean_Wind_Speed=("Wind_Speed", "mean"),
        Max_Wind_Speed=("Wind_Speed", "max")
    )
    .reset_index()
)

wind_daily["Date"] = pd.to_datetime(wind_daily["Date"])

print("Shape:", wind_daily.shape)
display(wind_daily.head())

Shape: (2192, 5)


,Date,Mean_U10,Mean_V10,Mean_Wind_Speed,Max_Wind_Speed
0,2015-01-01,-0.970402,-0.688204,1.757655,2.625952
1,2015-01-02,-1.682431,-0.455764,1.946573,3.647534
2,2015-01-03,0.962810,-0.866344,1.492209,2.766795
3,2015-01-04,1.773298,-1.831503,2.604788,3.700177
4,2015-01-05,4.105233,-2.425379,4.797141,7.469673


### Findings

The Wind Dataset was successfully aggregated to a daily temporal resolution, resulting in **2,192** daily observations for the Delhi study region. Daily mean values of the horizontal wind components (`Mean_U10` and `Mean_V10`), along with the daily mean and maximum wind speeds, were retained to preserve the observed wind characteristics. The aggregated dataset is now consistent with the daily temporal resolution required for integration with the Air Pollution, Fire, and Weather Datasets.

### 7.6 Verifying the Study Period

The temporal coverage of the aggregated Wind Dataset is verified to ensure consistency with the study period and compatibility with the remaining datasets prior to integration.

In [91]:
print("Earliest Date :", wind_daily["Date"].min())
print("Latest Date   :", wind_daily["Date"].max())

Earliest Date : 2015-01-01 00:00:00
Latest Date   : 2020-12-31 00:00:00


### Findings

The aggregated Wind Dataset covers the period from **1 January 2015** to **31 December 2020**, providing complete daily wind information for the entire study duration. The consistent temporal coverage ensures that the wind data can be effectively integrated with the Air Pollution, Fire, and Weather Datasets during the dataset merging stage.

## 8. Weather Dataset Preprocessing

This section prepares the Weather Dataset for integration with the remaining datasets. The preprocessing includes examining the dataset structure, selecting the variables required for the study, and aggregating the observations to a daily temporal resolution.

### 8.1 Examining the Dataset Structure

The Weather Dataset is first examined to verify its dimensions, variable names, data types, and identify the meteorological variables available for analysis.

In [92]:
print("Shape:", weather_df.shape)

display(weather_df.head())

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nData Types:")
print(weather_df.dtypes)

Shape: (2955892, 9)


,time,latitude,longitude,number,step,surface,valid_time,d2m,t2m
0,2015-01-01 00:00:00,32.0,74.00,0,0 days,0.0,2015-01-01 00:00:00,279.61612,279.87158
1,2015-01-01 00:00:00,32.0,74.25,0,0 days,0.0,2015-01-01 00:00:00,279.54190,280.00635
2,2015-01-01 00:00:00,32.0,74.50,0,0 days,0.0,2015-01-01 00:00:00,280.18057,280.45752
3,2015-01-01 00:00:00,32.0,74.75,0,0 days,0.0,2015-01-01 00:00:00,280.68057,281.25440
4,2015-01-01 00:00:00,32.0,75.00,0,0 days,0.0,2015-01-01 00:00:00,280.62198,281.27002



Columns:
['time', 'latitude', 'longitude', 'number', 'step', 'surface', 'valid_time', 'd2m', 't2m']

Data Types:
time           object
latitude      float64
longitude     float64
number          int64
step           object
surface       float64
valid_time     object
d2m           float64
t2m           float64
dtype: object


### 8.2 Selecting Required Variables

The Weather Dataset contains several metadata fields generated during the ERA5 reanalysis process. Since these variables do not represent meteorological observations and are not required for the objectives of this study, they are removed. Only the observation time, geographic coordinates, and weather measurements required for subsequent analysis are retained.

In [93]:
weather_df = weather_df[
    ["valid_time", "latitude", "longitude", "t2m", "d2m"]
].copy()

print("Shape:", weather_df.shape)

display(weather_df.head())

Shape: (2955892, 5)


,valid_time,latitude,longitude,t2m,d2m
0,2015-01-01 00:00:00,32.0,74.00,279.87158,279.61612
1,2015-01-01 00:00:00,32.0,74.25,280.00635,279.54190
2,2015-01-01 00:00:00,32.0,74.50,280.45752,280.18057
3,2015-01-01 00:00:00,32.0,74.75,281.25440,280.68057
4,2015-01-01 00:00:00,32.0,75.00,281.27002,280.62198


### Findings
The Weather Dataset originally contained 2,955,892 observations and nine variables. After removing ERA5 metadata variables (number, step, surface, and time), five variables were retained: valid_time, latitude, longitude, t2m (2 m air temperature), and d2m (2 m dew point temperature). These variables contain the temporal, spatial, and meteorological information required for the study.

### 8.3 Selecting the Nearest Grid Point to Delhi

The Weather Dataset contains observations from multiple grid cells across northern India. Since the study focuses on Delhi, only the nearest ERA5 grid point to Delhi is retained for subsequent analysis.

In [94]:
# Delhi coordinates
delhi_lat = 28.6
delhi_lon = 77.2

# Calculate distance from every grid point to Delhi
grid_points = weather_df[['latitude', 'longitude']].drop_duplicates().copy()

grid_points['distance'] = (
    (grid_points['latitude'] - delhi_lat) ** 2 +
    (grid_points['longitude'] - delhi_lon) ** 2
)

nearest = grid_points.loc[grid_points['distance'].idxmin()]

print("Nearest Weather Grid Point")
print(nearest)

weather_delhi = weather_df[
    (weather_df['latitude'] == nearest['latitude']) &
    (weather_df['longitude'] == nearest['longitude'])
].copy()

print("\nShape:", weather_delhi.shape)

display(weather_delhi.head())

Nearest Weather Grid Point
latitude     28.5000
longitude    77.2500
distance      0.0125
Name: 251, dtype: float64

Shape: (10228, 5)


,valid_time,latitude,longitude,t2m,d2m
251,2015-01-01 00:00:00,28.5,77.25,285.23682,282.47745
540,2015-01-01 06:00:00,28.5,77.25,291.46260,282.67017
829,2015-01-01 12:00:00,28.5,77.25,291.56232,285.52300
1118,2015-01-01 18:00:00,28.5,77.25,287.59448,284.83264
1407,2015-01-02 00:00:00,28.5,77.25,288.60920,286.55700


### Findings
The nearest ERA5 weather grid to Delhi was identified at latitude 28.50° and longitude 77.25°. Filtering to this location reduced the dataset from 2,955,892 observations to 10,228 six-hourly weather records, corresponding to the study area. This ensures that all subsequent analyses use meteorological conditions representative of Delhi.

### 8.4 Converting Data Types and Temperature Units

The observation timestamp is converted to a datetime format to enable temporal aggregation. Air temperature and dew point temperature are converted from Kelvin to degrees Celsius to improve interpretability while preserving the original measurements.

In [95]:
weather_delhi = weather_delhi.copy()

# Convert timestamp to datetime
weather_delhi["time"] = pd.to_datetime(weather_delhi["valid_time"])

# Convert Kelvin to Celsius
weather_delhi["temp_C"] = weather_delhi["t2m"] - 273.15
weather_delhi["dewpoint_C"] = weather_delhi["d2m"] - 273.15

# Remove columns no longer needed
weather_delhi = weather_delhi.drop(columns=["valid_time", "t2m", "d2m"])

print(weather_delhi.dtypes)
print()
display(weather_delhi.head())

latitude             float64
longitude            float64
time          datetime64[ns]
temp_C               float64
dewpoint_C           float64
dtype: object



,latitude,longitude,time,temp_C,dewpoint_C
251,28.5,77.25,2015-01-01 00:00:00,12.08682,9.32745
540,28.5,77.25,2015-01-01 06:00:00,18.31260,9.52017
829,28.5,77.25,2015-01-01 12:00:00,18.41232,12.37300
1118,28.5,77.25,2015-01-01 18:00:00,14.44448,11.68264
1407,28.5,77.25,2015-01-02 00:00:00,15.45920,13.40700


### Findings
The timestamp was converted to datetime format to support temporal operations. Air temperature and dew point temperature were converted from Kelvin to degrees Celsius. The converted values fall within realistic ranges for Delhi, confirming that the unit conversion was successful.

### 8.5 Assessing Data Quality

The filtered weather dataset is assessed for missing values and duplicate observations before aggregation. Identifying these issues at this stage ensures that daily weather summaries are computed from complete and reliable observations.

In [96]:
print("Shape:", weather_delhi.shape)

print("\nMissing values:")
print(weather_delhi.isna().sum())

print("\nDuplicate rows:", weather_delhi.duplicated().sum())

print("\nUnique timestamps:", weather_delhi["time"].nunique())

Shape: (10228, 5)

Missing values:
latitude      0
longitude     0
time          0
temp_C        0
dewpoint_C    0
dtype: int64

Duplicate rows: 0

Unique timestamps: 10228


### Findings
The filtered weather dataset contains no missing values or duplicate observations. Each six-hourly timestamp is unique, indicating that the dataset is complete and suitable for aggregation into daily weather summaries without requiring additional data cleaning.

### 8.6 Aggregating Weather Observations to Daily Level

The ERA5 weather observations are available at six-hour intervals. To align the temporal resolution with the Air Quality, Fire, and Wind datasets, the weather observations are aggregated to daily values by calculating the mean daily air temperature and dew point temperature.

In [97]:
# Create date column
weather_delhi["Date"] = weather_delhi["time"].dt.normalize()

# Aggregate to daily values
weather_daily = (
    weather_delhi
    .groupby("Date", as_index=False)
    .agg(
        Avg_Temperature_C=("temp_C", "mean"),
        Avg_DewPoint_C=("dewpoint_C", "mean")
    )
)

print("Shape:", weather_daily.shape)

display(weather_daily.head())

Shape: (2557, 3)


,Date,Avg_Temperature_C,Avg_DewPoint_C
0,2015-01-01,15.814055,10.725815
1,2015-01-02,15.319465,13.485620
2,2015-01-03,15.595728,14.065458
3,2015-01-04,13.921083,11.270265
4,2015-01-05,14.195888,9.797250


### Findings
The six-hourly ERA5 weather observations were aggregated to daily values by calculating the mean daily air temperature and mean daily dew point temperature. This reduced the dataset from 10,228 observations to 2,557 daily records, matching the temporal resolution required for integration with the Air Quality, Fire, and Wind datasets.

### 8.7 Calculating Daily Relative Humidity

Relative humidity is calculated from the daily mean air temperature and daily mean dew point temperature using the Magnus equation. This approach provides a physically meaningful estimate of atmospheric moisture and is commonly used in meteorological and environmental studies.

In [98]:
import numpy as np

weather_daily["Relative_Humidity"] = (
    100 *
    np.exp((17.625 * weather_daily["Avg_DewPoint_C"]) /
           (243.04 + weather_daily["Avg_DewPoint_C"])) /
    np.exp((17.625 * weather_daily["Avg_Temperature_C"]) /
           (243.04 + weather_daily["Avg_Temperature_C"]))
)

# Round for readability
weather_daily["Relative_Humidity"] = weather_daily["Relative_Humidity"].round(2)

display(weather_daily.head())

,Date,Avg_Temperature_C,Avg_DewPoint_C,Relative_Humidity
0,2015-01-01,15.814055,10.725815,71.76
1,2015-01-02,15.319465,13.485620,88.82
2,2015-01-03,15.595728,14.065458,90.61
3,2015-01-04,13.921083,11.270265,84.05
4,2015-01-05,14.195888,9.797250,74.85


### Findings
Daily relative humidity was calculated from the mean daily air temperature and mean daily dew point temperature using the Magnus equation. The resulting humidity values fall within realistic atmospheric ranges, confirming that the derived weather variable accurately represents daily moisture conditions for the study area.

### 8.8 Final Quality Assessment

The final daily weather dataset is validated by checking for missing values, duplicate dates, and ensuring that the calculated relative humidity values lie within the physically valid range of 0–100%.

In [99]:
print("Shape:", weather_daily.shape)

print("\nMissing Values")
print(weather_daily.isnull().sum())

print("\nDuplicate Dates:", weather_daily["Date"].duplicated().sum())

print("\nRelative Humidity Range")
print("Minimum:", weather_daily["Relative_Humidity"].min())
print("Maximum:", weather_daily["Relative_Humidity"].max())

display(weather_daily.head())

Shape: (2557, 4)

Missing Values
Date                 0
Avg_Temperature_C    0
Avg_DewPoint_C       0
Relative_Humidity    0
dtype: int64

Duplicate Dates: 0

Relative Humidity Range
Minimum: 20.53
Maximum: 98.72


,Date,Avg_Temperature_C,Avg_DewPoint_C,Relative_Humidity
0,2015-01-01,15.814055,10.725815,71.76
1,2015-01-02,15.319465,13.485620,88.82
2,2015-01-03,15.595728,14.065458,90.61
3,2015-01-04,13.921083,11.270265,84.05
4,2015-01-05,14.195888,9.797250,74.85


### Findings
The final daily weather dataset contains 2,557 observations with no missing values or duplicate dates. The calculated relative humidity ranges from 20.53% to 98.72%, which falls within the physically valid range for atmospheric humidity. These results confirm that the weather preprocessing pipeline produced a complete, reliable, and analysis-ready dataset for integration with the Air Quality, Fire, and Wind datasets.

### Weather Dataset Preprocessing Summary

The ERA5 weather dataset was successfully preprocessed for the Delhi Air Quality study. Metadata variables unrelated to meteorological observations were removed, and the nearest ERA5 grid cell representing Delhi (28.50°N, 77.25°E) was selected. Air temperature and dew point temperature were converted from Kelvin to degrees Celsius after filtering the study location. The six-hourly observations were then aggregated into daily mean values to match the temporal resolution of the Air Quality, Fire, and Wind datasets. Daily relative humidity was subsequently derived using the Magnus equation based on the daily mean air temperature and dew point temperature. Comprehensive data quality assessment confirmed the absence of missing values and duplicate observations, while all derived humidity values were within physically valid limits. The resulting dataset is clean, consistent, and suitable for integration into the final modelling dataset.

### 9. Saving the Cleaned Datasets

After completing the preprocessing of the Air Quality, Fire, Wind, and Weather datasets, each cleaned dataset is saved as a separate comma-separated values (CSV) file. Saving these datasets preserves the preprocessing results, improves reproducibility, and allows subsequent stages of the study, including data integration and predictive modelling, to be performed without repeating the preprocessing pipeline.

In [100]:
# Save cleaned datasets

aqi_df.to_csv("aqi_cleaned.csv", index=False)

fire_daily.to_csv("fire_cleaned.csv", index=False)

wind_daily.to_csv("wind_cleaned.csv", index=False)

weather_daily.to_csv("weather_cleaned.csv", index=False)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.


### Findings
The preprocessing outputs were successfully exported as separate CSV files. Maintaining individual cleaned datasets ensures reproducibility of the preprocessing workflow, facilitates future modifications to individual data sources, and provides reusable inputs for dataset integration and subsequent modelling tasks.

# 10. Master Dataset Creation

### 10.1 Integrating the Preprocessed Datasets

Following preprocessing, the Air Quality, Fire, Wind, and Weather datasets are integrated into a single master dataset using the common **Date** attribute. Combining these datasets creates a unified analytical dataset that contains pollution measurements, fire activity, meteorological conditions, and wind characteristics for each day of the study period. This integrated dataset serves as the foundation for exploratory data analysis, feature engineering, and predictive modelling.

In [101]:
# Merge all cleaned datasets

master_df = (
    aqi_df
    .merge(fire_daily, on="Date", how="inner")
    .merge(wind_daily, on="Date", how="inner")
    .merge(weather_daily, on="Date", how="inner")
)

print("Shape:", master_df.shape)

display(master_df.head())


Shape: (1861, 15)


,Date,AQI,AQI_Bucket,PM2.5,Fire_Count,Total_FRP,Mean_FRP,Max_FRP,Mean_U10,Mean_V10,Mean_Wind_Speed,Max_Wind_Speed,Avg_Temperature_C,Avg_DewPoint_C,Relative_Humidity
0,2015-01-01,472.0,Severe,313.22,30,129.63,4.321000,11.74,-0.970402,-0.688204,1.757655,2.625952,15.814055,10.725815,71.76
1,2015-01-02,454.0,Severe,186.18,15,84.92,5.661333,16.66,-1.682431,-0.455764,1.946573,3.647534,15.319465,13.485620,88.82
2,2015-01-03,143.0,Moderate,87.18,4,5.93,1.482500,2.07,0.962810,-0.866344,1.492209,2.766795,15.595728,14.065458,90.61
3,2015-01-04,319.0,Very Poor,151.84,15,81.51,5.434000,15.46,1.773298,-1.831503,2.604788,3.700177,13.921083,11.270265,84.05
4,2015-01-05,325.0,Very Poor,146.60,34,110.50,3.250000,8.84,4.105233,-2.425379,4.797141,7.469673,14.195888,9.797250,74.85


In [105]:
master_df.isnull().sum()

Date                 0
AQI                  9
AQI_Bucket           9
PM2.5                1
Fire_Count           0
Total_FRP            0
Mean_FRP             0
Max_FRP              0
Mean_U10             0
Mean_V10             0
Mean_Wind_Speed      0
Max_Wind_Speed       0
Avg_Temperature_C    0
Avg_DewPoint_C       0
Relative_Humidity    0
dtype: int64

### 10.2 Handling Missing Values

The integrated master dataset contains a small number of missing observations in the AQI and PM2.5 variables. Since these variables are essential for the research objectives and subsequent predictive modeling, records containing missing values are removed. Given that fewer than 1% of the observations are affected, their removal has a negligible impact on the overall dataset while ensuring data quality for subsequent analyses.

In [106]:
master_df = master_df.dropna(
    subset=["AQI", "PM2.5"]
).reset_index(drop=True)

print("Shape:", master_df.shape)

print("\nRemaining Missing Values")
print(master_df.isnull().sum())

Shape: (1851, 15)

Remaining Missing Values
Date                 0
AQI                  0
AQI_Bucket           0
PM2.5                0
Fire_Count           0
Total_FRP            0
Mean_FRP             0
Max_FRP              0
Mean_U10             0
Mean_V10             0
Mean_Wind_Speed      0
Max_Wind_Speed       0
Avg_Temperature_C    0
Avg_DewPoint_C       0
Relative_Humidity    0
dtype: int64


### Findings

After removing records containing missing AQI and PM2.5 values, the master dataset contains **1,851 complete daily observations** with no remaining missing values. The cleaned dataset is now suitable for exploratory data analysis, feature engineering, hypothesis testing, and predictive modeling.

In [107]:
master_df.to_csv("master_dataset.csv", index=False)

print("master_dataset.csv saved successfully!")

master_dataset.csv saved successfully!
